# Bedrock에서 Claude 3 Haiku 파인튜닝하기
이 노트북에서는 Amazon Bedrock에서 Claude 3 Haiku를 파인튜닝하는 과정을 안내합니다

## 필요한 것
- Bedrock에 접근할 수 있는 AWS 계정
- 데이터셋 (또는 여기 제공된 샘플 데이터셋 사용)
- [학습 데이터를 저장할 s3 버킷에 접근할 수 있는 서비스 역할](https://docs.aws.amazon.com/bedrock/latest/userguide/model-customization-iam-role.html)

## 의존성 설치

In [ ]:
!pip install boto3

In [ ]:
import boto3

## 데이터셋 준비하기
Bedrock 파인튜닝용 데이터셋은 JSONL 파일이어야 합니다(즉 각 줄에 JSON 객체가 하나씩 있는 파일).

JSONL 파일의 각 줄은 다음 구조의 JSON 객체여야 합니다.

```
{
  "system": "<optional_system_message>",
  "messages": [
    {"role": "user", "content": "user message"},
    {"role": "assistant", "content": "assistant response"},
    ...
  ]
}
```

- `system` 필드는 선택 사항입니다.
- 메시지가 최소 두 개는 있어야 합니다.
- 첫 메시지는 "user"에서 와야 합니다.
- 마지막 메시지는 "assistant"에서 와야 합니다.
- user와 assistant 메시지가 번갈아 나와야 합니다.
- 그 밖의 키는 허용되지 않습니다.


## 샘플 데이터셋 — JSON 모드
모든 질문에 JSON으로 답하도록 모델을 가르치는 샘플 데이터셋을 포함해 두었습니다. 그 데이터셋은 다음과 같습니다:

In [ ]:
import json

sample_dataset = []
dataset_path = "datasets/json_mode_dataset.jsonl"
with open(dataset_path) as f:
    for line in f:
        sample_dataset.append(json.loads(line))

print(json.dumps(sample_dataset[0], indent=2))

## 데이터셋을 S3에 업로드하기
파인튜닝용 데이터셋은 s3에 있어야 합니다. 이 데모에서는 여러분이 관리하는 s3 버킷에 샘플 데이터셋을 씁니다

In [ ]:
bucket_name = "YOUR_BUCKET_NAME"
s3_path = "json_mode_dataset.jsonl"

s3 = boto3.client("s3")
s3.upload_file(dataset_path, bucket_name, s3_path)

## Bedrock 파인튜닝 작업 시작하기

데이터셋을 준비했으니 `boto3`로 파인튜닝 작업을 시작할 수 있습니다. 먼저 작업에 필요한 몇 가지 파라미터를 설정합니다:

In [ ]:
# Configuration
job_name = "anthropic-finetuning-cookbook-training"
custom_model_name = "anthropic_finetuning_cookbook"
role = "YOUR_AWS_SERVICE_ROLE_ARN"
output_path = f"s3://{bucket_name}/finetuning_example_results/"
base_model_id = (
    "arn:aws:bedrock:us-east-1::foundation-model/anthropic.claude-haiku-4-5-20251001-v1:0:200k"
)

# Hyperparameters
epoch_count = 5
batch_size = 4
learning_rate_multiplier = 1.0

그런 다음 `boto3`로 작업을 시작합니다

In [ ]:
bedrock = boto3.client(service_name="bedrock")
bedrock_runtime = boto3.client(service_name="bedrock-runtime")

bedrock.create_model_customization_job(
    customizationType="FINE_TUNING",
    jobName=job_name,
    customModelName=custom_model_name,
    roleArn=role,
    baseModelIdentifier=base_model_id,
    hyperParameters={
        "epochCount": f"{epoch_count}",
        "batchSize": f"{batch_size}",
        "learningRateMultiplier": f"{learning_rate_multiplier}",
    },
    trainingDataConfig={"s3Uri": f"s3://{bucket_name}/{s3_path}"},
    outputDataConfig={"s3Uri": output_path},
)

학습이 진행되는 동안 다음으로 작업 상태를 확인할 수 있습니다:

In [ ]:
# Check for the job status
status = bedrock.get_model_customization_job(jobIdentifier=job_name)["status"]

## 파인튜닝한 모델 사용하기!

파인튜닝한 모델을 사용하려면 [Amazon Bedrock의 프로비저닝된 처리량으로 호스팅해야 합니다](https://docs.aws.amazon.com/bedrock/latest/userguide/model-customization-use.html). 프로비저닝된 처리량으로 모델이 준비되면 Bedrock API로 모델을 호출할 수 있습니다.

In [ ]:
provisioned_throughput_arn = "YOUR_PROVISIONED_THROUGHPUT_ARN"

In [ ]:
bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")
body = json.dumps(
    {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1000,
        "system": "JSON Mode: Enabled",
        "messages": [
            {
                "role": "user",
                "content": [{"type": "text", "text": "What is a large language model?"}],
            }
        ],
    }
)
response = bedrock_runtime.invoke_model(modelId=provisioned_throughput_arn, body=body)
body = json.loads(response["body"].read().decode("utf-8"))

In [ ]:
print(body["content"][0]["text"])